# First Agent with smolagents — 데스크탑 Ollama판

Dummy Agent 노트북에서는 **손으로** 했어요: system prompt 쓰기 → `stop`으로 끊기 → JSON 꺼내기 → 도구 실행 → 관찰 붙여서 다시 부르기 → `while` 루프.
이번엔 그걸 **smolagents 라이브러리가 대신** 해 줘요. 우리가 할 일은 딱 3개예요.

| 우리가 할 일 | 코드 |
|---|---|
| ① 모델(두뇌) 고르기 | `LiteLLMModel(...)` |
| ② 도구 만들기 | `@tool` 붙인 파이썬 함수 |
| ③ 에이전트 만들고 일 시키기 | `CodeAgent(tools=..., model=...)` → `agent.run("질문")` |

**원문(코스)과 다른 점** — 원문은 HF Space(인터넷 서버)에서 HF 유료 모델을 쓰는데, 우리는 크레딧이 0이라 **내 데스크탑의 Ollama**로 바꿨어요.

| 원문 | 이 노트북 | 이유 |
|---|---|---|
| HF Space를 복제해서 `app.py` 수정 | 이 노트북에서 한 칸씩 | 같은 코드를 내 PC에서 실행 |
| `InferenceClientModel(model_id='Qwen/Qwen2.5-Coder-32B-Instruct')` | `LiteLLMModel(model_id="ollama_chat/qwen3.5:9b")` | HF 서버 대신 내 PC의 Ollama |
| `prompts.yaml`을 읽어서 넣음 | 안 넣음 (smolagents 기본값) | 템플릿의 yaml은 옛 버전(1.13)용이라 지금 버전과 형식이 안 맞음 |
| 이미지 생성 도구 `load_tool(...)` | 뺌 | HF 서버(크레딧)를 쓰고, 남의 코드를 내려받아 실행(`trust_remote_code`)해야 함 |
| `GradioUI(agent).launch()` | 8단계(선택), `share=False` | 원문의 채팅 화면을 내 PC에서만 띄움 |

위에서부터 `Shift + Enter`로 한 칸씩 실행하세요.

## 0단계: 준비 확인

Ollama가 켜져 있는지, smolagents가 설치됐는지 봐요.

In [ ]:
import json
import urllib.request

import smolagents

with urllib.request.urlopen("http://localhost:11434/api/tags", timeout=10) as resp:
    models = [m["name"] for m in json.loads(resp.read())["models"]]

print("smolagents 버전:", smolagents.__version__)
print("설치된 모델:", models)
print("qwen3.5:9b 있음?", "qwen3.5:9b" in models)

✅ **확인할 것**: `smolagents 버전: 1.26.0`, `qwen3.5:9b 있음? True`
- `ModuleNotFoundError: No module named 'smolagents'` → PowerShell에서 `pip install -r requirements.txt`를 안 했거나, venv가 꺼진 상태에서 Jupyter를 켬
- `ConnectionRefusedError` → Ollama 앱이 꺼져 있음

## 1단계: 모델 연결 (두뇌 고르기)

Dummy 때 만든 `chat()` 함수(편지 봉투에 담아 Ollama에 보내고 답장 본문만 꺼내기)를 **smolagents가 `LiteLLMModel`로 대신** 해 줘요.
- **LiteLLM** = 여러 회사 AI(OpenAI, Anthropic, Ollama …)를 **같은 말투로 부르게 해 주는 통역사** 라이브러리. `model_id` 앞의 `ollama_chat/`가 "Ollama에 채팅 방식으로 보내라"는 뜻
- 원문의 `InferenceClientModel`은 "HF 서버로 보내라". **모델 칸만 바꾸면 나머지 코드는 그대로** — 이게 프레임워크의 장점

| 설정 | 뜻 | Dummy 때 |
|---|---|---|
| `api_base` | Ollama 주소 | `OLLAMA_URL` |
| `num_ctx=8192` | 모델이 한 번에 읽을 수 있는 길이(컨텍스트 창). smolagents의 system prompt가 길어서(약 8,000자) 넉넉히 | (기본값 썼음) |
| `reasoning_effort="none"` | 속생각 끄기 | `"think": False` |

In [ ]:
from smolagents import LiteLLMModel

model = LiteLLMModel(
    model_id="ollama_chat/qwen3.5:9b",
    api_base="http://localhost:11434",
    num_ctx=8192,
    reasoning_effort="none",
)

# smolagents의 메시지 꼴: content가 "조각 목록" (글 조각, 이미지 조각 …을 섞을 수 있게)
messages = [{"role": "user", "content": [{"type": "text", "text": "The capital of France is"}]}]
reply = model(messages)
print(reply.content)

✅ **확인할 것**: Paris가 들어간 답. 첫 호출은 모델을 GPU에 올리느라 수십 초 걸릴 수 있어요.
- Dummy 1단계와 **같은 질문, 같은 답** — 바뀐 건 우리가 `chat()`을 직접 안 짰다는 것뿐
- 빨간 글씨 `Provider List: https://docs.litellm.ai/...` 줄이 같이 나오면 LiteLLM의 안내문이라 무시해도 돼요 (답이 나왔으면 성공)

## 2단계: 도구 만들기 (`@tool`)

원문의 `get_current_time_in_timezone`을 그대로 써요. 함수 위에 `@tool` 한 줄만 붙이면 smolagents 도구가 돼요.

원문이 강조한 두 가지:
1. **입력·출력 타입**: `(timezone: str) -> str` — "글자를 받아서 글자를 돌려준다"
2. **형식 맞춘 docstring**(함수 밑 `"""설명"""`): 첫 줄 = 도구 설명, `Args:` 아래 = 인자마다 설명

왜 중요하냐면, smolagents가 이 **타입과 설명을 읽어서 system prompt의 도구 설명을 자동으로 써 주기** 때문이에요. Dummy 2단계에서 `get_weather: Get the current weather...`를 손으로 쓴 그 부분이에요. (설명이 빠지면 smolagents가 에러를 냄)

In [ ]:
import datetime

import pytz
from smolagents import tool


@tool
def get_current_time_in_timezone(timezone: str) -> str:
    """A tool that fetches the current local time in a specified timezone.
    Args:
        timezone: A string representing a valid timezone (e.g., 'America/New_York').
    """
    try:
        # Create timezone object
        tz = pytz.timezone(timezone)
        # Get current time in that timezone
        local_time = datetime.datetime.now(tz).strftime("%Y-%m-%d %H:%M:%S")
        return f"The current local time in {timezone} is: {local_time}"
    except Exception as e:
        return f"Error fetching time for timezone '{timezone}': {str(e)}"


# @tool이 docstring에서 뽑아낸 정보
print("이름:", get_current_time_in_timezone.name)
print("설명:", get_current_time_in_timezone.description)
print("입력:", get_current_time_in_timezone.inputs)
print("출력:", get_current_time_in_timezone.output_type)

# 도구는 그냥 함수처럼 직접 불러 볼 수도 있어요
print()
print(get_current_time_in_timezone("Asia/Seoul"))

✅ **확인할 것**
- `설명`·`입력`에 docstring 내용이 들어가 있음 = 우리가 따로 안 써도 도구 설명서가 만들어짐
- 마지막 줄에 지금 서울 시각 (`Asia/Seoul`이 시간대 이름. `Europe/London`, `America/New_York`처럼 `대륙/도시` 꼴)

## 3단계: 에이전트 만들기 (`CodeAgent`)

`CodeAgent` = **행동(Action)을 파이썬 코드로 쓰는** 에이전트. Actions 섹션에서 배운 "코드 액션"이에요.
- Dummy: 모델이 JSON(`{"action": "get_weather", ...}`)을 쓰면 → 우리가 꺼내서 실행
- CodeAgent: 모델이 `get_current_time_in_timezone(timezone="Asia/Seoul")` 같은 **파이썬 코드**를 쓰면 → smolagents가 그 코드를 **실행**하고, `print()`로 찍힌 것을 관찰(Observation)로 돌려줌

| 설정 | 뜻 |
|---|---|
| `tools=[...]` | 쓸 수 있는 도구 목록. 원문은 `[final_answer]`만 있어서 "도구 없는 에이전트" → 우리가 채워 넣는 게 과제 |
| `max_steps=6` | 최대 6바퀴 (Dummy 7단계의 `max_steps`) |
| `verbosity_level=1` | 바퀴마다 무슨 일이 있었는지 화면에 보여 주기 |

원문은 `final_answer = FinalAnswerTool()`을 직접 넣었는데, 지금 버전은 **`final_answer` 도구를 자동으로 추가**해 줘요(아래에서 확인). `final_answer(...)` = Dummy의 `Final Answer:` 글자 대신 쓰는 "끝내기 버튼".

In [ ]:
from smolagents import CodeAgent

agent = CodeAgent(
    tools=[get_current_time_in_timezone],
    model=model,
    max_steps=6,
    verbosity_level=1,
)

print("에이전트가 가진 도구:", list(agent.tools))

✅ **확인할 것**: `['get_current_time_in_timezone', 'final_answer']` — 내가 안 넣은 `final_answer`가 들어 있음

### smolagents가 써 준 system prompt 엿보기

Dummy 2단계에서 손으로 쓴 "근무 매뉴얼"을 smolagents는 자동으로 써요. 앞부분과, 우리 도구가 소개된 부분만 봐요.

In [ ]:
sp = agent.system_prompt
print("system prompt 길이:", len(sp), "자\n")
print(sp[:900])
print("\n........ (중간 생략: 예시 여러 개) ........\n")
i = sp.index("def get_current_time_in_timezone")
print(sp[i - 200 : i + 500])

✅ **확인할 것**
- 앞부분: `Thought:` → `<code>`로 시작해 `</code>`로 끝나는 코드 → `Observation:` 사이클을 하라고 함 = Dummy의 Thought → Action → Observation 규칙과 같은 구조
- 뒷부분: 우리 도구가 `def get_current_time_in_timezone(timezone: string) -> string:` + docstring 꼴로 소개됨 = 2단계의 `@tool`이 만든 것
- Dummy의 system prompt는 약 1,500자였는데 이건 몇 배 길어요 (예시가 많음). 그래서 1단계에서 `num_ctx`를 늘렸어요

## 4단계: 첫 실행

질문 하나 던지고 지켜봐요. 바퀴(Step)마다 **모델이 쓴 코드** → **실행 결과(Execution logs)**가 차례로 나와요.

In [ ]:
answer = agent.run("What time is it now in Seoul?")
print("\n★ 최종 답:", answer)

✅ **확인할 것**
- `Step 1`: 모델이 `get_current_time_in_timezone(...)`을 부르는 코드를 씀 → `Execution logs:`에 진짜 시각 (= 관찰)
- `Step 2` (또는 같은 바퀴): `final_answer(...)` → `Final answer:` 줄
- Dummy 때 3~7단계에서 우리가 손으로 한 일이 **`agent.run` 한 줄**로 끝남
- 에러(`Error in code parsing` 등)가 나도 smolagents가 그 에러를 **관찰로 모델에게 돌려줘서** 다음 바퀴에 고치게 해요 (Dummy 7단계의 "친절한 에러 관찰"을 알아서 해 줌). 6바퀴 안에 끝나면 OK

## 5단계: 속 들여다보기 — Dummy와 비교

에이전트는 바퀴마다 한 일을 `agent.memory.steps`에 적어 둬요 (Dummy 7단계의 `messages` 대본 같은 것).

In [ ]:
for step in agent.memory.steps:
    if not hasattr(step, "code_action"):   # 맨 앞의 '질문(Task)' 기록은 건너뜀
        continue
    print(f"===== {step.step_number}바퀴 =====")
    print("[모델이 쓴 글 = Thought + 코드]")
    print((step.model_output or "").strip()[:600])
    print("[smolagents가 꺼내서 실행한 코드 = Action]")
    print(step.code_action)
    print("[실행 결과 = Observation]")
    print((step.observations or "").strip()[:300])
    if step.error:
        print("[에러]", step.error)
    print()

✅ **확인할 것**: 바퀴마다 Thought → 코드(Action) → 결과(Observation)가 보임

### 손으로 한 일 vs smolagents

| Dummy에서 손으로 한 일 | smolagents에서 |
|---|---|
| system prompt에 도구 설명·규칙 쓰기 | `@tool`의 docstring + 자동 system prompt (3단계) |
| `chat()` 함수 짜기 | `LiteLLMModel` (1단계) |
| `stop=["Observation:"]` | 자동. smolagents도 똑같이 `stop=["Observation:", "Calling tools:", "</code>"]`를 보냄 (클라우드 가짜 서버로 확인) |
| JSON 꺼내서 `TOOLS[이름](**인자)` | 모델이 쓴 **코드를 그대로 실행** |
| 관찰을 user 메시지로 붙이기 (6-B) | 자동. smolagents도 관찰을 **user 메시지**로 넣음 → 우리가 6-B로 고친 게 맞는 방향이었음 |
| 에러를 친절하게 알려 주기 | 자동 (에러 내용을 관찰로) |
| `for` 루프 + `Final Answer:` 찾기 | `max_steps` + `final_answer()` 도구 |

## 6단계: 내 도구 추가 — Dummy 7단계 질문 다시 하기

원문의 과제 = "**도구를 추가해서 에이전트에게 새 능력을 주기**". 원문의 `my_custom_tool`(아무것도 안 하는 빈 도구) 자리에, Dummy 때 만든 **가짜 날씨 도구**를 `@tool`로 바꿔 넣어 봐요.
그리고 Dummy 7단계에서 3바퀴 걸린 질문을 그대로 해 봐요.

In [ ]:
FAKE_WEATHER = {
    "London": "sunny with low temperatures, 8°C",
    "Paris": "rainy, 14°C",
    "Seoul": "clear and cold, 2°C",
}


@tool
def get_weather(location: str) -> str:
    """Get the current weather in a given city.
    Args:
        location: The city name in English, e.g. 'London'.
    """
    return FAKE_WEATHER.get(location, f"Error: unknown location '{location}'. Known cities: {list(FAKE_WEATHER)}")


agent = CodeAgent(
    tools=[get_weather, get_current_time_in_timezone],
    model=model,
    max_steps=6,
    verbosity_level=1,
)

answer = agent.run("What is the weather of London and Paris? Which city is warmer?")
print("\n★ 최종 답:", answer)

✅ **확인할 것**
- Paris(14°C)가 더 따뜻하다는 답
- **몇 바퀴 걸렸나?** Dummy(JSON 액션)는 한 바퀴에 도구를 하나씩 불러서 3바퀴였어요. 코드 액션은 한 코드 안에서 `get_weather("London")`, `get_weather("Paris")`를 **둘 다** 부를 수 있어서 더 빨리 끝날 수 있어요 (Actions 섹션의 "코드 액션 장점"). 모델이 어떻게 쓰는지 보세요
- 비교 숫자(14 > 8)를 Thought로 했는지, 코드(`if`)로 했는지도 보세요

## 7단계: 웹 검색 도구 추가 (원문에 import 돼 있던 것)

원문 첫 줄에 들어 있던 `DuckDuckGoSearchTool` = 인터넷 검색 도구. 이번엔 가짜가 아니라 **진짜 바깥 세상**에서 정보를 가져와요 (데스크탑 인터넷 필요, 토큰 불필요).

In [ ]:
from smolagents import DuckDuckGoSearchTool

search_tool = DuckDuckGoSearchTool(max_results=3)

agent = CodeAgent(
    tools=[search_tool, get_current_time_in_timezone],
    model=model,
    max_steps=6,
    verbosity_level=1,
)

answer = agent.run("Search the web: what is the Hugging Face smolagents library? Answer in 2 sentences.")
print("\n★ 최종 답:", answer)

✅ **확인할 것**
- 코드에 `web_search(query=...)`가 보이고, 관찰에 검색 결과(제목·링크·요약)가 나옴
- 최종 답이 검색 결과에 근거했는지 (지어낸 게 아닌지) 관찰과 비교해 보세요
- `RatelimitException`·`DDGSException` 같은 에러 = 검색 사이트가 잠깐 막은 것. 1~2분 뒤 다시 실행 (클라우드에서는 네트워크가 막혀 이 단계는 미리 확인 못 했어요)

## 8단계 (선택): 원문의 채팅 화면(Gradio) 띄우기

원문 마지막 줄 `GradioUI(agent).launch()` = 에이전트를 **채팅 웹페이지**로 감싸 주는 것. Space에서 보이는 화면이 이거예요. 우리는 내 PC에서 띄워요.
- ⚠️ **`share=False` 꼭 넣기**: smolagents의 기본값은 `share=True` = 인터넷에 **누구나 들어올 수 있는 공개 링크**를 만들어요. 이 에이전트는 모델이 쓴 코드를 **내 PC에서 실행**하니까, 내 PC에서만 열리게 막아요 (원문은 Space라서 HF 서버 안에서 돌았던 것)
- 실행하면 칸 아래에 `Running on local URL:  http://127.0.0.1:7860`이 나와요 → 그 주소를 **새 브라우저 탭**에 붙여 넣기
- 이 칸은 끝나지 않고 계속 돌아요(칸 왼쪽 `[*]`). 정상이에요 — 채팅 화면을 띄워 두는 중
- 끝낼 때: 노트북 위쪽 ■(정지) 버튼, 또는 메뉴 **Kernel → Interrupt**

In [ ]:
from smolagents import GradioUI

GradioUI(agent).launch(share=False)   # share=False: 내 PC에서만 열림

✅ **확인할 것**: 브라우저의 채팅창에 질문을 쓰면 바퀴(Step)마다 생각·코드·결과가 접혀서 보이고 마지막에 답이 나옴

## 더 해 볼 실험 (선택)
- 4단계 질문을 `"What time is it in New York and in Seoul? What is the time difference?"`로 → 시간 계산을 코드로 하나?
- 6단계에서 `"What's the weather in Tokyo?"` → 가짜 날씨에 없는 도시. 에러 관찰을 보고 어떻게 하나? (Dummy 때와 비교)
- 나만의 도구 만들기: 아무 파이썬 함수에 `@tool` + docstring. 예) 두 날짜 사이 날짜 수 세기, 주사위 굴리기

## 다 했으면
4·6·7단계 결과(몇 바퀴 돌았나, 모델이 쓴 코드, 최종 답, 에러가 있었다면 그 내용)를 복사해서 AI에게 붙여 주세요 → 노트 10번에 기록해요.